In [5]:
#if (!require("BiocManager", quietly = TRUE))
    #install.packages("BiocManager")

#BiocManager::install("MPRAnalyze")
#Format the data into MPRA project
library("MPRAnalyze")
library("BiocParallel")

register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)

In [2]:
process_datasets <- function(dna_file, rna_file, annot_file, neg_control_file) {
 
  # Function to drop enhancer or any specific column and set row names
  drop_and_rename <- function(df, col_name) {
    if (col_name %in% names(df)) {
      if (any(is.na(df[[col_name]])) || any(duplicated(df[[col_name]]))) {
        stop(paste("Column", col_name, "contains NA or duplicate values."))
      }
      row.names(df) <- df[[col_name]]
      df <- df[, !(names(df) %in% c(col_name))]
    }
    df
  }

  # Load and process CSV files
  load_and_process <- function(filepath, drop_func, col_name) {
    df <- read.csv(filepath, header = TRUE)
    drop_func(df, col_name)
  }

  # Load data
  df_DNA <- load_and_process(dna_file, drop_and_rename, "enhancer_id")
  df_RNA <- load_and_process(rna_file, drop_and_rename, "enhancer_id")
  annot_DNA <- load_and_process(annot_file, drop_and_rename, "X")

  # Calculate selected columns for subsetting
  total_columns <- ncol(df_DNA)
  selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

  # Subset data starting from the 32nd row
  df_DNA <- df_DNA[32:nrow(df_DNA), selected_columns]
  df_RNA <- df_RNA[32:nrow(df_RNA), selected_columns]
  annot_DNA <- annot_DNA[selected_columns,]

  # Convert columns to factor for MPRAnalyze
  annot_DNA[] <- lapply(annot_DNA, as.factor)

  # Load negative controls and extract sequence IDs
  negative <- read.csv(neg_control_file, sep = "\t", header = FALSE)
  control <- as.character(negative$V1)  # V1 is the sequence_ID
  
  # Return a list of processed data frames and control
  list(DNA = as.matrix(df_DNA), RNA = as.matrix(df_RNA), Annotation = annot_DNA, Control = control)
}


In [3]:
df_DNA <- read.csv("read_counts_R1R2/HMC3_DNA_matched_barcodes_reshaped.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/HMC3_RNA_matched_barcodes_reshaped.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_barcodes.csv", header=TRUE,row.names=1)

df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

df_DNA <- df_DNA[32:nrow(df_DNA), selected_columns]
df_RNA <- df_RNA[32:nrow(df_RNA), selected_columns]
annot_DNA <- annot_DNA[selected_columns,]


#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String,
                          reducedDesign = ~ Tissue,
                          #mode="scale"
                          )                     
res <-testLrt(obj)
write.csv(res,"20240812_comparative_HMC3_allele.csv")                  

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-14 01:08:35.575431
Success: TRUE

Task duration:
   user  system elapsed 
184.428   1.359 201.220 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6156563 328.8   11606425 619.9  7494129 400.3
Vcells 12133629  92.6   23107961 176.3 19189968 146.5

Log messages:
INFO [2024-08-14 01:05:14] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 13
Node: 12
Timestamp: 2024-08-14 01:09:54.72843
Success: TRUE

Task duration:
   user  system elapsed 
264.961   1.135 280.693 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6157910 328.9   11606425 619.9  7494129 400.3
Vcells 12139705  92.7   23107961 176.3 19189968 146.5

Log messages:
INFO [2024-08-14 01:05:14] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 11
Node: 14
Timestamp: 2024-08-14 01:

In [3]:
# Example of how to call the function
results <- process_datasets(
    "read_counts_R1R2/HMC3_IFNB_DNA_matched_barcodes_reshaped.csv",
    "read_counts_R1R2/HMC3_IFNB_RNA_matched_barcodes_reshaped.csv",
    "annotation_barcodes/mpra3_annot_HMC3_IFNB_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240129_comparative_HMC3_IFNB_allele.csv")



#################################################################################################
# Example of how to call the function
results <- process_datasets(
    "read_counts_R1R2/HMC3_LPSIFNG_DNA_matched_barcodes_reshaped.csv",
    "read_counts_R1R2/HMC3_LPSIFNG_RNA_matched_barcodes_reshaped.csv",
    "annotation_barcodes/mpra3_annot_HMC3_IFNB_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240129_comparative_HMC3_LPSIFNG_allele.csv")

#################################################################################################
# Example of how to call the function
results <- process_datasets(
    "read_counts_R1R2/HMC3_Naive_DNA_matched_barcodes_reshaped.csv",
    "read_counts_R1R2/HMC3_Naive_RNA_matched_barcodes_reshaped.csv",
    "annotation_barcodes/mpra3_annot_HMC3_Naive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
  #                control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240129_comparative_HMC3_Naive_allele.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-01-30 15:56:19.12606
Success: TRUE

Task duration:
   user  system elapsed 
 34.986   1.157  39.468 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6288995 335.9   11968425 639.2  7237810 386.6
Vcells 11940084  91.1   23100310 176.3 23100279 176.3

Log messages:
INFO [2024-01-30 15:55:39] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 1
Node: 24
Timestamp: 2024-01-30 15:56:31.397487
Success: TRUE

Task duration:
   user  system elapsed 
 49.893   0.852  52.258 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6289448 335.9   11968425 639.2  7237810 386.6
Vcells 11944913  91.2   23100310 176.3 23100279 176.3

Log messages:
INFO [2024-01-30 15:55:39] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 7
Node: 18
Timestamp: 2024-01-30 15:56

In [3]:
df_DNA <- read.csv("read_counts_R1R2/HMC3_IFNG_DNA_matched_barcodes_reshaped.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/HMC3_IFNG_RNA_matched_barcodes_reshaped.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_IFNG_barcodes.csv", header=TRUE,row.names=1)

df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

df_DNA <- df_DNA[32:nrow(df_DNA), selected_columns]
df_RNA <- df_RNA[32:nrow(df_RNA), selected_columns]
annot_DNA <- annot_DNA[selected_columns,]

obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
  #                control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240813_comparative_HMC3_IFNG_allele.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 23:09:25.813099
Success: TRUE

Task duration:
   user  system elapsed 
 31.818   0.421  38.411 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6156072 328.8   11606415 619.9  7494129 400.3
Vcells 11220258  85.7   19189968 146.5 19172266 146.3

Log messages:
INFO [2024-08-13 23:08:47] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 23
Node: 2
Timestamp: 2024-08-13 23:09:40.357772
Success: TRUE

Task duration:
   user  system elapsed 
 46.473   0.357  53.006 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6157416 328.9   11606415 619.9  7494129 400.3
Vcells 11225555  85.7   19189968 146.5 19172266 146.3

Log messages:
INFO [2024-08-13 23:08:47] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 18
Node: 7
Timestamp: 2024-08-13 23:0

Load experiment

 if there is any ALT-specific interaction in the interferon treatments

In [4]:

df_DNA <- read.csv("read_counts_R1R2/HMC3_IFNBvsNaive_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/HMC3_IFNBvsNaive_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_IFNBvsNaive_barcodes.csv")
df_DNA<-df_DNA[32:nrow(df_DNA), ]
df_RNA<-df_RNA[32:nrow(df_RNA), ]

#Match row names
annot_DNA<-dropX(annot_DNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
     #             control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNB_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240129_comparative_HMC3_IFNBvsNaive_interaction.csv")




df_DNA <- read.csv("read_counts_R1R2/HMC3_LPSIFNGvsNaive_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/HMC3_LPSIFNGvsNaive_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_LPSIFNGvsNaive_barcodes.csv")
df_DNA<-df_DNA[32:nrow(df_DNA), ]
df_RNA<-df_RNA[32:nrow(df_RNA), ]

#Match row names
annot_DNA<-dropX(annot_DNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
             #     control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240129_comparative_HMC3_LPSIFNGvsNaive_interaction.csv")



Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-01-30 16:04:11.236348
Success: TRUE

Task duration:
   user  system elapsed 
 70.571   1.454  74.490 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6290034 336.0   11968425 639.2  7350826 392.6
Vcells 12653503  96.6   23100310 176.3 23100307 176.3

Log messages:
INFO [2024-01-30 16:02:56] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 12
Node: 13
Timestamp: 2024-01-30 16:04:42.385962
Success: TRUE

Task duration:
   user  system elapsed 
104.021   1.153 106.032 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6290483 336.0   11968425 639.2  7350826 392.6
Vcells 12658712  96.6   23100310 176.3 23100307 176.3

Log messages:
INFO [2024-01-30 16:02:56] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 14
Node: 11
Timestamp: 2024-01-30 16

In [11]:


df_DNA <- read.csv("read_counts_R1R2/HMC3_LPSIFNGvsIFNG_DNA_matched_barcodes_reshaped.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/HMC3_LPSIFNGvsIFNG_RNA_matched_barcodes_reshaped.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_LPSIFNGvsIFNG_barcodes.csv", header=TRUE,row.names=1)

df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

df_DNA <- df_DNA[32:nrow(df_DNA), selected_columns]
df_RNA <- df_RNA[32:nrow(df_RNA), selected_columns]
annot_DNA <- annot_DNA[selected_columns,]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
    #              control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240813_comparative_HMC3_LPSIFNGvsIFNG_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 23:19:53.832342
Success: TRUE

Task duration:
   user  system elapsed 
 60.117   0.951  67.916 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6172136 329.7   11606415 619.9 10904013 582.4
Vcells 11522940  88.0   19189968 146.5 19189968 146.5

Log messages:
INFO [2024-08-13 23:18:46] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 7
Node: 18
Timestamp: 2024-08-13 23:20:18.711671
Success: TRUE

Task duration:
   user  system elapsed 
 86.543   0.844  93.436 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6172588 329.7   11606415 619.9 10904013 582.4
Vcells 11526606  88.0   19189968 146.5 19189968 146.5

Log messages:
INFO [2024-08-13 23:18:45] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 14
Node: 11
Timestamp: 2024-08-13 23:

In [12]:
df_DNA <- read.csv("read_counts_R1R2/HMC3_IFNGvsNaive_DNA_matched_barcodes_reshaped.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/HMC3_IFNGvsNaive_RNA_matched_barcodes_reshaped.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_IFNGvsNaive_barcodes.csv", header=TRUE,row.names=1)

df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

df_DNA <- df_DNA[32:nrow(df_DNA), selected_columns]
df_RNA <- df_RNA[32:nrow(df_RNA), selected_columns]
annot_DNA <- annot_DNA[selected_columns,]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
    #              control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240813_comparative_HMC3_IFNGvsNaive_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 23:23:15.833709
Success: TRUE

Task duration:
   user  system elapsed 
 69.730   1.017  77.791 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6172310 329.7   11606415 619.9 10904013 582.4
Vcells 11587592  88.5   19189968 146.5 19189968 146.5

Log messages:
INFO [2024-08-13 23:21:58] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 6
Node: 19
Timestamp: 2024-08-13 23:23:49.346849
Success: TRUE

Task duration:
   user  system elapsed 
103.568   1.014 111.961 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6172762 329.7   11606415 619.9 10904013 582.4
Vcells 11591301  88.5   19189968 146.5 19189968 146.5

Log messages:
INFO [2024-08-13 23:21:57] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 4
Node: 21
Timestamp: 2024-08-13 23:2

# Motif-Allele Comparison

In [2]:
df_DNA <- read.csv("read_counts_R1R2/HMC3_DNA_matched_barcodes_reshaped_motif.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/HMC3_RNA_matched_barcodes_reshaped_motif.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_barcodes_motif.csv", header=TRUE,row.names=1)

df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Test,
                          rnaDesign = ~ Motif_Allele+Tissue,
                          reducedDesign = ~ Tissue,
                          #mode="scale"
                          )
                      
res <-testLrt(obj)
write.csv(res,"20240812_comparative_HMC3_motif.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 11:14:01.039405
Success: TRUE

Task duration:
   user  system elapsed 
 18.721   0.208  20.395 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6153420 328.7   11606445 619.9  7569850 404.3
Vcells 11289936  86.2   19189968 146.5 19189853 146.5

Log messages:
INFO [2024-08-13 11:13:40] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 22
Node: 3
Timestamp: 2024-08-13 11:14:35.877367
Success: TRUE

Task duration:
   user  system elapsed 
 52.069   0.599  55.319 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6154404 328.7   11606445 619.9  7569850 404.3
Vcells 11292939  86.2   19189968 146.5 19189853 146.5

Log messages:
INFO [2024-08-13 11:13:40] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 13
Node: 12
Timestamp: 2024-08-13 11:

# Allele Only

In [1]:
process_datasets <- function(dna_file, rna_file, annot_file, neg_control_file) {
  # Set multicore parameters
  register(MulticoreParam(24, log = TRUE, stop.on.error = FALSE))
  
  # Function to drop enhancer or any specific column and set row names
  drop_and_rename <- function(df, col_name) {
    if (col_name %in% names(df)) {
      if (any(is.na(df[[col_name]])) || any(duplicated(df[[col_name]]))) {
        stop(paste("Column", col_name, "contains NA or duplicate values."))
      }
      row.names(df) <- df[[col_name]]
      df <- df[, !(names(df) %in% c(col_name))]
    }
    df
  }

  # Load and process CSV files
  load_and_process <- function(filepath, drop_func, col_name) {
    df <- read.csv(filepath, header = TRUE)
    drop_func(df, col_name)
  }

  # Load data
  df_DNA <- load_and_process(dna_file, drop_and_rename, "enhancer_id")
  df_RNA <- load_and_process(rna_file, drop_and_rename, "enhancer_id")
  annot_DNA <- load_and_process(annot_file, drop_and_rename, "X")

  # Calculate selected columns for subsetting
  total_columns <- ncol(df_DNA)
  selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

  # Subset data starting from the 32nd row
  df_DNA <- df_DNA[, selected_columns]
  df_RNA <- df_RNA[, selected_columns]
  annot_DNA <- annot_DNA[selected_columns,]

  # Convert columns to factor for MPRAnalyze
  annot_DNA[] <- lapply(annot_DNA, as.factor)

  # Load negative controls and extract sequence IDs
  negative <- read.csv(neg_control_file, sep = "\t", header = FALSE)
  control <- as.character(negative$V1)  # V1 is the sequence_ID
  
  # Return a list of processed data frames and control
  list(DNA = as.matrix(df_DNA), RNA = as.matrix(df_RNA), Annotation = annot_DNA, Control = control)
}


In [8]:
df_DNA <- read.csv("read_counts_R1R2/allele_only/HMC3_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/HMC3_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_barcodes.csv", header=TRUE,row.names=1)

df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]


#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String,
                          reducedDesign = ~ Tissue,
                          #mode="scale"
                          )                     
res <-testLrt(obj)
write.csv(res,"20240812_comparative_HMC3_alleleOnly.csv")                  
    

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 00:40:47.312928
Success: TRUE

Task duration:
   user  system elapsed 
170.172   1.137 182.614 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6172900 329.7   11606428 619.9  8718485 465.7
Vcells 11991620  91.5   23107961 176.3 23107961 176.3

Log messages:
INFO [2024-08-13 00:37:44] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 8
Node: 17
Timestamp: 2024-08-13 00:41:41.917867
Success: TRUE

Task duration:
   user  system elapsed 
220.350   1.281 237.680 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6174176 329.8   11606428 619.9  8718485 465.7
Vcells 11997030  91.6   23107961 176.3 23107961 176.3

Log messages:
INFO [2024-08-13 00:37:44] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 6
Node: 19
Timestamp: 2024-08-13 00:4

In [17]:
df_DNA <- read.csv("read_counts_R1R2/allele_only/HMC3_IFNG_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/HMC3_IFNG_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_IFNG_barcodes.csv", header=TRUE,row.names=1)
df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
##############################################################
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,

                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )

                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240715_comparative_HMC3_IFNG_alleleOnly.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 4
Node: 21
Timestamp: 2024-07-17 00:45:15.148422
Success: TRUE

Task duration:
   user  system elapsed 
 24.792   0.304  25.134 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6357113 339.6   12116344 647.1 11709118 625.4
Vcells 11766160  89.8   23202795 177.1 23202608 177.1

Log messages:
INFO [2024-07-17 00:44:50] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 5
Node: 20
Timestamp: 2024-07-17 00:45:16.069651
Success: TRUE

Task duration:
   user  system elapsed 
 25.318   0.296  25.698 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6358187 339.6   12116344 647.1 11709118 625.4
Vcells 11771694  89.9   23202795 177.1 23202608 177.1

Log messages:
INFO [2024-07-17 00:44:50] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 23
Node: 2
Timestamp: 2024-07-17 00:4

In [8]:
# Example of how to call the function
results <- process_datasets(
    "read_counts_R1R2/allele_only/HMC3_IFNB_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/HMC3_IFNB_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_HMC3_IFNB_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )
                                   
res <-testLrt(obj)
write.csv(res,"20240616_comparative_HMC3_IFNB_allele.csv")



# Example of how to call the function
results <- process_datasets(
    "read_counts_R1R2/allele_only/HMC3_LPSIFNG_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/HMC3_LPSIFNG_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_HMC3_LPSIFNG_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          )
                       
res <-testLrt(obj)
write.csv(res,"20240616_comparative_HMC3_LPSIFNG_allele.csv")

# Example of how to call the function
results <- process_datasets(
    "read_counts_R1R2/allele_only/HMC3_Naive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/HMC3_Naive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_HMC3_Naive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )
                        
res <-testLrt(obj)
write.csv(res,"20240616_comparative_HMC3_Naive_allele.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-06-18 00:29:08
Success: TRUE

Task duration:
   user  system elapsed 
 31.291   0.184 818.047 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5821646 311.0    9412435 502.7  8934276 477.2
Vcells 11284591  86.1   18899734 144.2 18877097 144.1

Log messages:
INFO [2024-06-18 00:15:31] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 1
Node: 24
Timestamp: 2024-06-18 00:32:22
Success: TRUE

Task duration:
    user   system  elapsed 
  41.114    0.267 1022.695 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5822293 311.0    9412435 502.7  8934276 477.2
Vcells 11287997  86.2   18899734 144.2 18877097 144.1

Log messages:
INFO [2024-06-18 00:15:22] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 18
Node: 7
Timestamp: 2024-06-18 00:32:34
Suc

In [2]:
df_DNA <- read.csv("read_counts_R1R2/allele_only/HMC3_IFNGvsNaive_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/HMC3_IFNGvsNaive_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_IFNGvsNaive_barcodes.csv", header=TRUE,row.names=1)

df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )

obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )
                     
res <-testLrt(obj)
write.csv(res,"20240812_comparative_HMC3_IFNGvsNaive_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 11:10:28.583246
Success: TRUE

Task duration:
   user  system elapsed 
 65.116   0.612  68.973 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6155204 328.8   11606445 619.9  7494129 400.3
Vcells 11449381  87.4   19189968 146.5 19189968 146.5

Log messages:
INFO [2024-08-13 11:09:19] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 4
Node: 21
Timestamp: 2024-08-13 11:10:50.607596
Success: TRUE

Task duration:
   user  system elapsed 
 85.526   0.618  91.539 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6156478 328.8   11606445 619.9  7494129 400.3
Vcells 11454470  87.4   19189968 146.5 19189968 146.5

Log messages:
INFO [2024-08-13 11:09:19] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 14
Node: 11
Timestamp: 2024-08-13 11:

In [15]:
df_DNA <- read.csv("read_counts_R1R2/allele_only/HMC3_LPSIFNGvsIFNG_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/HMC3_LPSIFNGvsIFNG_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_LPSIFNGvsIFNG_barcodes.csv", header=TRUE,row.names=1)

df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )
                     
res <-testLrt(obj)
write.csv(res,"20240812_comparative_HMC3_LPSIFNGvsIFNG_interaction.csv")


Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 11:06:56.845533
Success: TRUE

Task duration:
   user  system elapsed 
 56.592   0.918  63.898 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6178481 330.0   11606428 619.9 10553896 563.7
Vcells 11455456  87.4   23107961 176.3 23107961 176.3

Log messages:
INFO [2024-08-13 11:05:53] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 16
Node: 9
Timestamp: 2024-08-13 11:07:11.018646
Success: TRUE

Task duration:
   user  system elapsed 
 70.686   0.951  78.378 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6178854 330.0   11606428 619.9 10553896 563.7
Vcells 11458519  87.5   23107961 176.3 23107961 176.3

Log messages:
INFO [2024-08-13 11:05:52] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 11
Node: 14
Timestamp: 2024-08-13 11:

In [9]:


# Example of how to call the function
results <- process_datasets(
    "read_counts_R1R2/allele_only/HMC3_LPSIFNGvsNaive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/HMC3_LPSIFNGvsNaive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_HMC3_LPSIFNGvsNaive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
    #              control=control,
                  BPPARAM = bpparam
                  )
                  
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )
                     
res <-testLrt(obj)
write.csv(res,"20240616_comparative_HMC3_LPSIFNGvsNaive_interaction.csv")

# Example of how to call the function
results <- process_datasets(
    "read_counts_R1R2/allele_only/HMC3_IFNBvsNaive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/HMC3_IFNBvsNaive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_HMC3_IFNBvsNaive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
    #              control=control,
                  BPPARAM = bpparam
                  )
                  
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNB_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )
                     
res <-testLrt(obj)
write.csv(res,"20240616_comparative_HMC3_IFNBvsNaive_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-06-18 03:24:41
Success: TRUE

Task duration:
    user   system  elapsed 
 381.139 1637.430 4059.269 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5826527 311.2    9412435 502.7  9412435 502.7
Vcells 11576971  88.4   22759680 173.7 18899734 144.2

Log messages:
INFO [2024-06-18 02:17:04] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 8
Node: 17
Timestamp: 2024-06-18 03:42:16
Success: TRUE

Task duration:
    user   system  elapsed 
 489.912 2094.995 5127.006 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5826909 311.2    9412435 502.7  9412435 502.7
Vcells 11580121  88.4   22759680 173.7 18899734 144.2

Log messages:
INFO [2024-06-18 02:16:51] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 21
Node: 4
Timestamp: 2024-06-18 03:43: